In [ ]:
!pip install librosa numpy scikit-learn matplotlib

In [ ]:
import librosa
import soundfile as sf
y, sr = librosa.load(librosa.ex('trumpet'))

sf.write("sample_happy.wav", y[:sr*3], sr)       # First 3 sec
sf.write("sample_neutral.wav", y[sr*3:sr*6], sr)   # Middle 3 sec
sf.write("sample_sad.wav", y[sr*6:sr*9], sr)       # Last 3 sec

downloaded_files = ["sample_happy.wav", "sample_neutral.wav", "sample_sad.wav"]
print("Successfully created local audio samples:")
for f in downloaded_files:
    print(" -", f)

Successfully created local audio samples:
 - sample_happy.wav
 - sample_neutral.wav
 - sample_sad.wav


In [ ]:
import librosa
import numpy as np

def extract_features(audio_file):

    y, sr = librosa.load(audio_file, sr=None)


    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_values = []
    for t in range(pitches.shape[1]):
        idx = magnitudes[:, t].argmax()
        p = pitches[idx, t]
        if p > 0:
            pitch_values.append(p)

    mean_pitch = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_range = (max(pitch_values) - min(pitch_values)) if len(pitch_values) > 0 else 0

    rms = librosa.feature.rms(y=y)[0]
    mean_energy = np.mean(rms)

    return [mean_pitch, pitch_range, mean_energy]

print("Extracting features from generated samples...")
real_features = []
for file in downloaded_files:
    feats = extract_features(file)
    real_features.append(feats)
    print(f"{file} -> Pitch: {feats[0]:.1f} Hz, Range: {feats[1]:.1f} Hz, Energy: {feats[2]:.4f}")

Extracting features from generated samples...
sample_happy.wav -> Pitch: 1075.9 Hz, Range: 915.4 Hz, Energy: 0.0929
sample_neutral.wav -> Pitch: 759.3 Hz, Range: 3402.1 Hz, Energy: 0.0008
sample_sad.wav -> Pitch: 0.0 Hz, Range: 0.0 Hz, Energy: 0.0000


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


np.random.seed(42)


happy_data = np.random.normal(loc=[350, 200, 0.08], scale=[30, 20, 0.015], size=(100, 3))


neutral_data = np.random.normal(loc=[250, 100, 0.04], scale=[20, 15, 0.010], size=(100, 3))

sad_data = np.random.normal(loc=[150, 50, 0.015], scale=[20, 10, 0.005], size=(100, 3))


X = np.vstack((happy_data, neutral_data, sad_data))
y = np.array(["Happy/Excited"]*100 + ["Neutral/Calm"]*100 + ["Sad/Tired"]*100)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"Model Training Complete!")
print(f"Training Accuracy: {acc * 100:.1f}%")

Model Training Complete!
Training Accuracy: 100.0%


In [ ]:
# Test predictions on created sample audio files
for file in downloaded_files:
    feats = extract_features(file)
    prediction = model.predict([feats])[0]
    probs = model.predict_proba([feats])[0]

    print(f"\nFile: {file}")
    print(f"Predicted Emotion: -> {prediction.upper()} <-")
    for emotion, prob in zip(model.classes_, probs):
        print(f"  - {emotion}: {prob * 100:.1f}%")


File: sample_happy.wav
Predicted Emotion: -> HAPPY/EXCITED <-
  - Happy/Excited: 100.0%
  - Neutral/Calm: 0.0%
  - Sad/Tired: 0.0%

File: sample_neutral.wav
Predicted Emotion: -> HAPPY/EXCITED <-
  - Happy/Excited: 64.0%
  - Neutral/Calm: 30.0%
  - Sad/Tired: 6.0%

File: sample_sad.wav
Predicted Emotion: -> SAD/TIRED <-
  - Happy/Excited: 0.0%
  - Neutral/Calm: 0.0%
  - Sad/Tired: 100.0%


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=0
  warnings.warn(


In [ ]:
import os
import google.colab.files as files

print("Upload your audio file (.wav or .mp3):")
uploaded = files.upload()

if len(uploaded) > 0:
    user_file = list(uploaded.keys())[0]

    user_feats = extract_features(user_file)
    user_pred = model.predict([user_feats])[0]
    user_probs = model.predict_proba([user_feats])[0]

    print("\n==========================================")
    print(f"ANALYSIS FOR: '{user_file}'")
    print("==========================================")
    print(f"Mean Pitch: {user_feats[0]:.1f} Hz")
    print(f"Pitch Range: {user_feats[1]:.1f} Hz")
    print(f"Mean Energy: {user_feats[2]:.4f}")
    print("------------------------------------------")
    print(f"PREDICTED EMOTION: -> {user_pred.upper()} <-")
    print("------------------------------------------")

    print("\nConfidence Scores:")
    for emotion, prob in zip(model.classes_, user_probs):
        print(f" - {emotion}: {prob * 100:.1f}%")
else:
    print("No file uploaded.")

Upload your audio file (.wav or .mp3):


Saving WhatsApp Audio 2026-08-13 at 8.03.30 PM.mpeg to WhatsApp Audio 2026-08-13 at 8.03.30 PM.mpeg

ANALYSIS FOR: 'WhatsApp Audio 2026-08-13 at 8.03.30 PM.mpeg'
Mean Pitch: 848.7 Hz
Pitch Range: 3822.2 Hz
Mean Energy: 0.0589
------------------------------------------
PREDICTED EMOTION: -> HAPPY/EXCITED <-
------------------------------------------

Confidence Scores:
 - Happy/Excited: 100.0%
 - Neutral/Calm: 0.0%
 - Sad/Tired: 0.0%
